# Energy Efficiency ANN

This notebook implements an Artificial Neural Network (ANN) to predict heating and cooling loads of buildings based on design parameters.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam, SGD

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

ImportError: Traceback (most recent call last):
  File "c:\Users\prana\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## 1. Data Preprocessing

In [ ]:
# Load dataset
file_path = 'ENB2012_data.xlsx'
df = pd.read_excel(file_path)

# Rename columns for clarity (optional but helpful)
column_names = {
    'X1': 'Relative_Compactness',
    'X2': 'Surface_Area',
    'X3': 'Wall_Area',
    'X4': 'Roof_Area',
    'X5': 'Overall_Height',
    'X6': 'Orientation',
    'X7': 'Glazing_Area',
    'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load',
    'Y2': 'Cooling_Load'
}
df = df.rename(columns=column_names)

# Display first few rows
print("First 5 rows:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Define Features (X) and Targets (Y)
X = df.drop(columns=['Heating_Load', 'Cooling_Load'])
Y = df[['Heating_Load', 'Cooling_Load']]

# Normalize data (MinMax Scaling)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

print(f"\nTraining shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

## 2. Build ANN Model (ReLU)

In [ ]:
def build_model(activation='relu', learning_rate=0.01):
    model = Sequential([
        Dense(64, input_dim=X_train.shape[1], activation=activation),
        Dense(32, activation=activation),
        Dense(2)  # Output layer: 2 neurons for Heating & Cooling Load
    ])
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

model_relu = build_model(activation='relu')
model_relu.summary()

## 3. Train Model

In [ ]:
history_relu = model_relu.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test), 
    epochs=2, 
    batch_size=32, 
    verbose=1
)

## 4. Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, title="Model"):
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- {title} Evaluation ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")
    
    return y_pred

y_pred_relu = evaluate_model(model_relu, X_test, y_test, title="ReLU Model")

## 5. Experiments: Tanh & Sigmoid

In [ ]:
# Tanh Model
model_tanh = build_model(activation='tanh')
history_tanh = model_tanh.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test), 
    epochs=2, 
    batch_size=32, 
    verbose=0
)

# Sigmoid Model
model_sigmoid = build_model(activation='sigmoid')
history_sigmoid = model_sigmoid.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test), 
    epochs=2, 
    batch_size=32, 
    verbose=0
)

print("Training complete for Tanh and Sigmoid models.")

## 6. Gradient Vanishing Explanation

**Sigmoid:** The derivative of the sigmoid function is at most 0.25. As we backpropagate through many layers, these small gradients are multiplied, causing the gradient to vanish towards the earlier layers. This slows down or stops learning.

**Tanh:** The derivative is at most 1.0, which is better than sigmoid, but it can still suffer from vanishing gradients if the inputs are large (saturation regions).

**ReLU:** The derivative is 1 for positive inputs, which helps mitigate the vanishing gradient problem, allowing deep networks to learn faster and more effectively.

## 7. Plots

In [ ]:
# Loss Curves
plt.figure(figsize=(12, 5))
plt.plot(history_relu.history['loss'], label='ReLU Train')
plt.plot(history_relu.history['val_loss'], label='ReLU Val')
plt.plot(history_tanh.history['loss'], label='Tanh Train')
plt.plot(history_sigmoid.history['loss'], label='Sigmoid Train')
plt.title('Model Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

# Predicted vs Actual (ReLU Model)
plt.figure(figsize=(12, 5))

# Heating Load
plt.subplot(1, 2, 1)
plt.scatter(y_test['Heating_Load'], y_pred_relu[:, 0], alpha=0.7)
plt.plot([y_test['Heating_Load'].min(), y_test['Heating_Load'].max()], 
         [y_test['Heating_Load'].min(), y_test['Heating_Load'].max()], 'r--')
plt.title('Heating Load: Actual vs Predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')

# Cooling Load
plt.subplot(1, 2, 2)
plt.scatter(y_test['Cooling_Load'], y_pred_relu[:, 1], alpha=0.7, color='orange')
plt.plot([y_test['Cooling_Load'].min(), y_test['Cooling_Load'].max()], 
         [y_test['Cooling_Load'].min(), y_test['Cooling_Load'].max()], 'r--')
plt.title('Cooling Load: Actual vs Predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')

plt.tight_layout()
plt.show()